# 07. Critic Evaluator (`Critic`) Demo

### Architecture Role & Dual-Stage Verification
The **Critic** enforces strict quantitative grounding before any research brief can be released:

1. **Deterministic Code Checks (No Hallucinations)**:
   - **Number Grounding**: Every numerical figure in the brief is verified against the raw tool data in `state.tools`.
   - **Section Completeness**: Verifies all required sections are present.
2. **LLM Multi-Dimensional Judge**:
   - Evaluates Grounding (30%), Coverage (20%), Balance (20%), Clarity (15%), and Actionability (15%).
   - Minimum passing threshold: Weighted score $\ge$ 8.0.

In [1]:
import sys
from pathlib import Path

# Resolve src directory
project_root = Path.cwd().parent if Path.cwd().name == "demo" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from research_agent.workflows.review_loop import (
    DIMENSION_WEIGHTS,
    check_grounding,
    check_sections,
    evidence_numbers,
    make_review_nodes,
)
from research_agent.agents.refiner import refiner_agent
from research_agent.agent_framework import GraphState
from research_agent.interfaces import get_llm

print(f"Critic Policy Dimension Weights: {DIMENSION_WEIGHTS}")

Critic Policy Dimension Weights: {'grounding': 0.3, 'coverage': 0.2, 'balance': 0.2, 'clarity': 0.15, 'actionability': 0.15}


## Stage 1: Deterministic Code Checks (Grounding & Sections)

In [2]:
# Sample brief text to evaluate
brief_sample = (
    "# Research Brief: NVDA\n\n"
    "## Executive Summary\nStrong operational momentum and market expansion.\n\n"
    "## 1. Fundamental & Earnings Performance\nRevenue $30.04B, gross margin 75.1%.\n\n"
    "## 2. Technical Profile & Market Momentum\nPrice $128.50, RSI 63.34.\n\n"
    "## 3. Catalysts, Macro Context & Risk Analysis\nBlackwell ramp on track; supply chain risks managed.\n\n"
    "## 4. Investment Stance & Outlook\nOverweight / Bullish.\n"
)

# Raw tool evidence stored in blackboard state
raw_tools = {
    "get_earnings_data": {"revenue": "$30.04B", "gross_margin": "75.1%"},
    "get_market_technicals": {"price": "$128.50", "rsi_14": 63.34},
}

evidence = evidence_numbers(raw_tools)
grounding_report = check_grounding(brief_sample, evidence)
missing_sections = check_sections(brief_sample)

print(f"Figures checked          : {grounding_report.checked}")
print(f"Unverified numbers count : {len(grounding_report.unverified)}")
print(f"Missing sections count   : {len(missing_sections)}")
print("Deterministic Code Check : PASSED")

Figures checked          : 4
Unverified numbers count : 0
Missing sections count   : 0
Deterministic Code Check : PASSED


## Stage 2: LLM Multi-Dimensional Evaluation via critic_node

In [3]:
llm = get_llm()
critic_node, _, _, _ = make_review_nodes(llm=llm, refiner=refiner_agent(llm=llm))

state = GraphState(
    question="Analyze NVDA",
    agents={"DraftWriter": brief_sample},
    tools=raw_tools,
)

critic_update = critic_node(state)
print(f"Review Status  : {critic_update.get('review_status')}")

if state.review.get("history"):
    latest_report = state.review["history"][-1]
    print(f"Scores         : {latest_report.get('scores')}")
    print(f"Weighted Score : {latest_report.get('weighted')}")
    print(f"Passed         : {latest_report.get('passed')}")
    print(f"Issues Found   : {len(latest_report.get('issues', []))}")

Review Status  : REVISE 5.0/10, 0 blockers
Scores         : {'coverage': 3.0, 'balance': 2.0, 'clarity': 5.0, 'actionability': 2.0, 'grounding': 10.0}
Weighted Score : 5.05
Passed         : False
Issues Found   : 5
